# 其他语言 OpenAI Whisper-large-v3 训练

## 其他语言统一配置

In [1]:
from pathlib import Path

LANG = "fr"  # 切换为 "de" 或 "nn"

LANG_SPECS = {
    "fr": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_fr"
            "/cv-corpus-27.0-2026-09-11/fr"
        ),
        "language": "french",
        "abbr": "fr",
    },
    "de": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_ge"
            "cv-corpus-27.0-2026-09-11/de"
        ),
        "language": "german",
        "abbr": "de",
    },
    "nn": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_nork"
            "/cv-corpus-27.0-2026-09-11/nn-NO"
        ),
        "language": "nynorsk",
        "abbr": "nn",
    },
}

CFG = LANG_SPECS[LANG]
CV_ROOT = CFG["cv_root"]
WHISPER_LANGUAGE = CFG["language"]
LANGUAGE_ABBR = CFG["abbr"]
TASK = "transcribe"

## 缓存与路径

In [2]:
from pathlib import Path
import os
import re

# Hugging Face 缓存放到外接盘，避免占用 C 盘
HF_ROOT = "/sigan/ai-ml/huggingface"
os.environ["HF_HOME"] = HF_ROOT
os.environ["HF_HUB_CACHE"] = f"{HF_ROOT}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_ROOT}/datasets"

# Common Voice 27 中文目录
CV_ROOT = CFG["cv_root"]
CLIPS_DIR = CV_ROOT / "clips"

# 如果解压完成后目录去掉了 .partial，把上面的路径改成实际路径
print(CV_ROOT)
print(CLIPS_DIR)

/home/sigan/datasets/common_voice_27_fr/cv-corpus-27.0-2026-09-11/fr
/home/sigan/datasets/common_voice_27_fr/cv-corpus-27.0-2026-09-11/fr/clips


## 加载 TSV 和音频

In [3]:
from datasets import DatasetDict, Audio, load_dataset

# 读取一个 split：TSV 提供文本和文件名，clips 提供真实音频
def load_split(tsv_name):
    ds = load_dataset(
        "csv",
        data_files=str(CV_ROOT / tsv_name),
        delimiter="\t",
        keep_default_na=False,
    )["train"]

    # TSV 中只有文件名，这里补成能访问的绝对路径
    ds = ds.map(
        lambda batch: {
            "audio": [
                str(CLIPS_DIR / filename)
                for filename in batch["path"]
            ]
        },
        batched=True,
    )

    # 把字符串路径变成可解码的 16kHz 音频对象
    ds = ds.cast_column(
        "audio",
        Audio(sampling_rate=16000),
    )

    # 保留 client_id 和 accents 便于核查说话人与地区
    return ds.select_columns(
        ["audio", "sentence", "client_id", "accents"]
    )


common_voice = DatasetDict({
    "train": load_split("train.tsv"),
    "validation": load_split("dev.tsv"),
    "test": load_split("test.tsv"),
})

print(common_voice)
print(common_voice["train"][0])

Map:   0%|          | 0/621982 [00:00<?, ? examples/s]

Map:   0%|          | 0/16207 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/16207 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 621982
    })
    validation: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 16207
    })
    test: Dataset({
        features: ['audio', 'sentence', 'client_id', 'accents'],
        num_rows: 16207
    })
})


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


{'audio': {'path': '/home/sigan/datasets/common_voice_27_fr/cv-corpus-27.0-2026-09-11/fr/clips/common_voice_fr_21917455.mp3', 'array': array([ 1.45519152e-10,  1.74622983e-10, -1.38243195e-10, ...,
        4.79687424e-06,  1.18551561e-05,  1.88128615e-05]), 'sampling_rate': 16000}, 'sentence': "Pendant l'occupation napoléonienne, l'église est pillée et la sacristie utilisée comme écurie.", 'client_id': 'a7c881996f1d72a5bd09e3dcc8aa9994b83ec84a4159253abaded666c911e49199db329e91bdb8554b9ac6d8de0857d1e42932be47806b36e824cfce0528aca7', 'accents': ''}


## 加载 Processor 并预处理

In [4]:
from transformers import AutoProcessor

MODEL_ID = "openai/whisper-large-v3"

# large-v3 使用 128 个 Mel 频带，必须使用它自己的 Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language=WHISPER_LANGUAGE,
    task=TASK,
)

tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor


def prepare_dataset(batch):
    audio = batch["audio"]

    # 音频转 Log-Mel 特征，large-v3 输出约 [128, 3000]
    batch["input_features"] = (
        feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
        )
        .input_features[0]
    )

    # 正确文本转 token ID，作为训练 labels
    batch["labels"] = tokenizer(
        batch["sentence"]
    ).input_ids

    return batch

# 在 common_voice.map() 前限制训练数据
MAX_TRAIN_SAMPLES = 20000

common_voice["train"] = (
    common_voice["train"]
    .shuffle(seed=42)
    .select(
        range(
            min(
                MAX_TRAIN_SAMPLES,
                len(common_voice["train"]),
            )
        )
    )
)

# 29,814 条全量训练数据都处理；外接盘较慢时把 num_proc 改成 2 或 1
tokenized_dataset = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice["train"].column_names,
    num_proc=4,
    writer_batch_size=100,
)

# 验证集有 10,679 条，每次生成评估太慢，固定抽 1,000 条
eval_dataset = (
    tokenized_dataset["validation"]
    .shuffle(seed=42)
    .select(
        range(
            min(
                300,
                len(tokenized_dataset["validation"]),
            )
        )
    )
)

# 用完整训练集训练，最后用完整测试集评估
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

print("train：", len(train_dataset))
print("validation subset：", len(eval_dataset))
print("test：", len(test_dataset))

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map (num_proc=4):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/16207 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/16207 [00:00<?, ? examples/s]

train： 20000
validation subset： 300
test： 16207


## Data Collator

这组目标模块会同时覆盖 Encoder、Decoder、交叉注意力和 FFN。没有加入 proj_out，因为它是语言模型输出头，参数量大，通常不值得用 LoRA 训练。

In [5]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import jiwer
import torch

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForSpeechSeq2Seq,
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


# 把不同长度的文本标签补齐，音频特征也用 Processor 对齐
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [
            {"input_features": x["input_features"]}
            for x in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": x["labels"]}
            for x in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        if (
            labels[:, 0]
            == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels,
        }


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

## 定义指标

In [6]:
import re
import unicodedata

import opencc

# 繁体转简体
_t2s = opencc.OpenCC("t2s")

# 统一常见中英文标点
_PUNCT_MAP = str.maketrans({
    ",": "，",
    "?": "？",
    "!": "！",
    ":": "：",
    ";": "；",
    "(": "（",
    ")": "）",
    "[": "【",
    "]": "】",
    "{": "｛",
    "}": "｝",
    "、": "，",
})

# 简繁转换与标点归一化函数
def normalize_chinese(text, remove_punctuation=False):
    """
    Chinese ASR 文本统一：
    1. 繁体 -> 简体
    2. 全角字符标准化
    3. 英文转小写
    4. 删除空格
    5. 统一常见标点
    6. 可选删除全部标点
    """
    if text is None:
        return ""

    # 先统一繁简体
    text = _t2s.convert(text)

    # 将全角字母、数字等统一为半角形式
    text = unicodedata.normalize("NFKC", text)

    # 英文大小写统一
    text = text.lower()

    # 中文 ASR 通常不依赖空格
    text = re.sub(r"\s+", "", text)

    # 英文句点不误伤小数，例如 3.14 保持不变
    text = re.sub(r"(?<!\d)\.(?!\d)", "。", text)

    # 统一中英文标点
    text = text.translate(_PUNCT_MAP)

    # 连续标点压缩成一个
    text = re.sub(r"，+", "，", text)
    text = re.sub(r"。+", "。", text)
    text = re.sub(r"？+", "？", text)
    text = re.sub(r"！+", "！", text)

    # 如果只比较识别内容，可以删除全部标点
    if remove_punctuation:
        text = re.sub(
            r"[，。？！：；、（）【】｛｝,\.\?!:;()\[\]{}]",
            "",
            text,
        )

    return text.strip()

In [7]:
def normalize_text(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def compute_metrics(result):
    pred_ids = result.predictions
    label_ids = result.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    predictions = [
        normalize_text(text)
        for text in tokenizer.batch_decode(
            pred_ids,
            skip_special_tokens=True,
        )
    ]

    references = [
        normalize_text(text)
        for text in tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
        )
    ]

    return {
        "wer": float(
            jiwer.wer(
                reference=references,
                hypothesis=predictions,
            )
        )
    }

## 定义模型训练配置

In [8]:
# 8-bit 加载基座，显存更适合 RTX 4080
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True
    ),
    device_map="auto",
)

# 训练时关闭 KV Cache，并清除推理阶段的固定解码限制
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 训练过程中的生成式评估也固定使用中文转写
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=WHISPER_LANGUAGE,
    task=TASK,
)
# 不加这一项，trainer.predict() 可能使用错误的语言任务标记
model.generation_config.forced_decoder_ids = (
    forced_decoder_ids
)

# 开启梯度检查点，进一步降低训练激活显存
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# 全量线性模块：注意力 Q/K/V/O 加 FFN fc1/fc2
target_modules = [
    "q_proj",
    "k_proj",
    
    # 之前 v3 full 修改全部注意力、交叉注意力和 FFN；
    # 更新范围太大，导致 decoder 生成重复；
    # q/v 更新范围更小，更适合在基座已经较强时做小幅适配。
    # "v_proj",
    # "out_proj",
    # "fc1",
    # "fc2",
]

# 只训练 LoRA 参数，冻结量化后的 Whisper 基座
lora_config = LoraConfig(
    r=8,
    # lora_alpha=32,
    lora_alpha=8,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,547,422,720 || trainable%: 0.2541102666503436


## 训练

In [10]:
OUTPUT_DIR = (
    f"/sigan/ai-ml/models/"
    f"whisper-large-v3-{LANGUAGE_ABBR}-lora"
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # RTX 4080 16GB：物理 batch 2，累积 8 次，有效 batch=16
    per_device_train_batch_size=2,
    # 有效 batch = 2 × 8 = 16
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    # 先完整训练 1 轮
    num_train_epochs=1,
    # 当前 V3 已预训练功能强大，很容易推理退化，步数小点方便随时保存
    # max_steps=1000,
    
    # 低学习率，避免破坏 v3 原有生成能力
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_torch",

    # RTX 4080 支持 FP16，混合精度训练应该开启
    fp16=True,
    bf16=False,

    # 每 1000 个优化步验证一次；日志不会自动保存模型
    # 每 250 步观察 CER 和保存 checkpoint
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    logging_steps=20,

    # 验证时真正生成文本，而不是只计算 teacher-forcing loss
    predict_with_generate=True,
    generation_max_length=225,

    # 用验证集 CER 选择效果最好的 checkpoint
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,

    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    tokenizer=processor.tokenizer,
)

In [11]:
# 如果已有 checkpoint，改为 trainer.train(resume_from_checkpoint=True)
trainer.train()
# trainer.train(resume_from_checkpoint=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Wer
500,0.233500,0.197204,0.099079
1000,0.181900,0.188067,0.102572


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=1250, training_loss=0.39209111595153806, metrics={'train_runtime': 12004.631, 'train_samples_per_second': 1.666, 'train_steps_per_second': 0.104, 'total_flos': 6.8131160064e+19, 'train_loss': 0.39209111595153806, 'epoch': 1.0})

## 保存

In [12]:
# 保存 LoRA adapter 和 Processor；完整 Whisper 基座不需要重复保存
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("已保存：", OUTPUT_DIR)
# trainer.save_model("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# processor.save_pretrained("/sigan/ai-ml/models/whisper-large-v3-zh-smoke")
# print("已保存：", "/sigan/ai-ml/models/whisper-large-v3-zh-smoke")

已保存： /sigan/ai-ml/models/whisper-large-v3-fr-lora


## 评估

In [13]:
# 验证集只用于调参，最终结果必须看没有参与训练的测试集
test_result = trainer.predict(
    test_dataset,
    metric_key_prefix="test",
    max_length=225,
    num_beams=1,
)

print(test_result.metrics)

{'test_loss': 0.270550400018692, 'test_wer': 0.1916298647937617, 'test_runtime': 30543.4996, 'test_samples_per_second': 0.531, 'test_steps_per_second': 0.265}
